# Vocal2BGM no Kaggle (T4 16 GB)

Versão do `02_vocal2bgm.ipynb` adaptada para rodar em **Kaggle Notebooks**.  
GPU T4 (16 GB VRAM) elimina a necessidade de `offload_to_cpu`, então a geração roda **15–30× mais rápido** do que na 1050 Ti local.

## Antes de rodar — checklist

1. **Settings** (painel direito do Kaggle):
   - **Accelerator**: `GPU T4 x2` ou `GPU T4` (suficiente)
   - **Internet**: `On` (precisamos pra `git clone` e download dos modelos)
2. **Dataset** anexado (painel direito → Add data):
   - Adicione seu dataset privado `music-ai-workflow-audio` contendo `vocals.wav`
   - Ele aparecerá em `/kaggle/input/music-ai-workflow-audio/`
3. **Run All** (ou execute célula a célula). Primeira execução demora ~15 min só por causa do download dos modelos ACE-Step (~14 GB).

## O que esse notebook faz

1. Clona o repo `music-ai-workflow` (público no GitHub) e o `ACE-Step-1.5`.
2. Instala dependências (nano-vllm local + ACE-Step editable + utilitários).
3. Configura caminhos do Kaggle (input/working).
4. Roda a geração via CLI do ACE-Step com `task_type="complete"`.
5. Salva resultado em `/kaggle/working/` (você baixa no final pelo painel de output).

## 1. Setup — clone do repo e do ACE-Step

In [ ]:
import os, sys, subprocess, time
from pathlib import Path

WORK = Path('/kaggle/working')
REPO_DIR = WORK / 'music-ai-workflow'
ACE_DIR  = WORK / 'ACE-Step-1.5'

# Clone do repo do projeto
if not REPO_DIR.exists():
    !git clone --depth 1 https://github.com/cicero-martins/music-ai-workflow.git {REPO_DIR}
else:
    print(f'✓ Repo já em {REPO_DIR}')

# Clone do ACE-Step
if not ACE_DIR.exists():
    !git clone --depth 1 https://github.com/ace-step/ACE-Step-1.5.git {ACE_DIR}
else:
    print(f'✓ ACE-Step já em {ACE_DIR}')

sys.path.insert(0, str(REPO_DIR))
print('✓ paths configurados')

## 2. Instalar dependências

Kaggle já vem com PyTorch + CUDA pré-instalados. Só instalamos o que falta:  
`nano-vllm` (path local do ACE-Step) + `ACE-Step` editável + `imageio-ffmpeg`.

In [ ]:
# nano-vllm — pacote local do ACE-Step (não está no PyPI)
!pip install -q {ACE_DIR}/acestep/third_parts/nano-vllm

# ACE-Step como editável
!pip install -q -e {ACE_DIR}

# imageio-ffmpeg (para utils carregar áudio se for MP4 — vocals.wav já está em WAV mas mantemos compat)
!pip install -q imageio-ffmpeg

print('✓ dependências instaladas')

## 3. Validação rápida de imports e GPU

In [ ]:
# Limpa qualquer estado parcial de imports anteriores
# (workaround para sys.modules em estado quebrado quando alguma célula tocou
#  'acestep' antes do pip install -e terminar — comum no Kaggle no primeiro run)
import sys
for _mod in list(sys.modules):
    if _mod.startswith('acestep'):
        del sys.modules[_mod]

import torch
from acestep.handler import AceStepHandler
from acestep.inference import GenerationParams

print('torch:', torch.__version__)
print('CUDA disponível:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'VRAM: {vram:.1f} GB')
    assert vram >= 14, 'GPU com menos de 14 GB — habilite o acelerador no painel direito (Settings → Accelerator → GPU T4)'
print('✓ ACE-Step importado')

## 4. Configuração da geração

Edite as variáveis abaixo conforme seu caso.

In [ ]:
# ── CONFIGURAÇÃO — edite aqui ─────────────────────────────────────────────────

# Caminho do vocals.wav dentro do dataset Kaggle anexado.
# Se o dataset se chama 'music-ai-workflow-audio' e o arquivo está na raiz, fica:
DATASET_SLUG = 'music-ai-workflow-audio'   # nome do dataset (sem prefixo do usuário)
VOCAL_FILE   = 'vocals.wav'                # nome do arquivo dentro do dataset
VOCAL_PATH   = Path('/kaggle/input') / DATASET_SLUG / VOCAL_FILE

# Descrição do arranjo (em inglês — modelo treinado predominantemente em EN)
CAPTION = (
    'gritty garage blues rock, fuzzed distorted electric guitar with bluesy bend '
    'riffs and slide accents, heavy pounding drums with cracking snare and stomping kick, '
    'tambourine, raw lo-fi analog production, mid-tempo groove around 110 bpm, '
    'dirty energetic and swampy, room reverb on the drums'
)

# Quais tracks gerar (não inclua 'vocals' — o vocal de entrada permanece)
COMPLETE_TRACKS = 'drums,bass,guitar'

# Parâmetros de geração — T4 16GB aguenta steps altos com folga
INFERENCE_STEPS = 32       # 32–50 é qualidade boa; 8 é só validação
GUIDANCE_SCALE  = 7.0
DURATION_SEC    = 30.0     # T4 lida tranquilo com 30s+; pode subir até a duração do vocal
SEED            = -1       # -1 = aleatório

NOME_SAIDA = 'nunca_e_sempre_arranjo'

# ─────────────────────────────────────────────────────────────────────────────

OUTPUT_DIR = WORK / 'output'
OUTPUT_DIR.mkdir(exist_ok=True)

assert VOCAL_PATH.exists(), (
    f'Vocal não encontrado em {VOCAL_PATH}.\n'
    'Confirme que você anexou o dataset Kaggle no painel direito (Add data) '
    'e que DATASET_SLUG e VOCAL_FILE estão corretos.'
)
print(f'✓ vocal de entrada: {VOCAL_PATH}  ({VOCAL_PATH.stat().st_size/1e6:.1f} MB)')
print(f'✓ output dir: {OUTPUT_DIR}')

## 5. Gerar TOML e invocar a CLI do ACE-Step

Sem `offload_to_cpu` — T4 16GB carrega o modelo base inteiro na VRAM.

In [ ]:
import toml, tempfile

config = {
    'project_root': str(ACE_DIR),
    'checkpoint_dir': str(ACE_DIR / 'checkpoints'),
    'config_path': 'acestep-v15-base',     # complete exige modelo base
    'device': 'auto',
    'offload_to_cpu': False,                # 16 GB cabe tranquilo, sem offload
    'backend': 'pt',                        # 'pt' = PyTorch; vllm requer setup extra

    'task_type': 'complete',
    'src_audio': str(VOCAL_PATH),
    'complete_tracks': COMPLETE_TRACKS,
    'caption': CAPTION,
    'lyrics': '',
    'instrumental': False,

    'inference_steps': INFERENCE_STEPS,
    'guidance_scale': GUIDANCE_SCALE,
    'duration': DURATION_SEC,
    'seed': SEED,
    'batch_size': 1,
    'audio_format': 'wav',

    'thinking': False,
    'use_cot_metas': False,
    'use_cot_caption': False,
    'use_cot_language': False,

    'save_dir': str(OUTPUT_DIR),
}

config_path = Path(tempfile.gettempdir()) / f'ace_config_{NOME_SAIDA}.toml'
with open(config_path, 'w', encoding='utf-8') as f:
    toml.dump(config, f)

print('--- TOML ---')
print(config_path.read_text(encoding='utf-8'))

In [ ]:
# Executar CLI — primeira execução baixa ~14 GB de modelos

cli_script = ACE_DIR / 'cli.py'
log_path = WORK / f'ace_log_{NOME_SAIDA}.txt'

arquivos_antes = set(OUTPUT_DIR.glob('*'))
t0 = time.time()

with open(log_path, 'w', encoding='utf-8', errors='replace') as f:
    resultado = subprocess.run(
        [sys.executable, str(cli_script), '-c', str(config_path)],
        stdout=f, stderr=subprocess.STDOUT, text=True,
    )

elapsed = time.time() - t0
arquivos_novos = sorted(set(OUTPUT_DIR.glob('*')) - arquivos_antes)

print(f'Exit code: {resultado.returncode}  |  Tempo: {elapsed:.1f}s')
print(f'Log completo em: {log_path}')

if arquivos_novos:
    print(f'\n✓ Geração OK — {len(arquivos_novos)} arquivo(s):')
    for p in arquivos_novos:
        print(f'   {p.name}  ({p.stat().st_size/1e6:.1f} MB)')
else:
    print('\n✗ Nenhum arquivo novo. Últimas 30 linhas do log:')
    print('-' * 60)
    print('\n'.join(log_path.read_text(encoding='utf-8', errors='replace').splitlines()[-30:]))

## 6. Tocar o resultado no notebook

In [ ]:
from IPython.display import Audio, display

if arquivos_novos:
    print('Entrada (vocal):')
    display(Audio(str(VOCAL_PATH)))
    print('\nSaída (arranjo gerado):')
    display(Audio(str(arquivos_novos[0])))
else:
    print('Nenhuma saída para tocar — veja erro acima.')

## 7. Download do resultado

Os arquivos em `/kaggle/working/output/` aparecem no painel direito → seção **Output** → você pode baixar individualmente.

**Quando terminar a sessão** (para liberar tempo da sua cota de 30h/semana):
- Menu superior → **Run** → **Stop session**, ou
- Feche a aba (sessão expira automaticamente após inatividade).

## Iteração — outros experimentos

Para gerar variações, edite a célula 4 (configuração) com novos valores e re-execute as células 5 e 6. Cada execução cria um arquivo novo em `/kaggle/working/output/` (sem sobrescrever).

Sugestões de variação:
- **Mudar caption** para outro estilo (ex: lo-fi indie, dream pop, alt rock)
- **Mudar `COMPLETE_TRACKS`** (ex: `'drums,bass'` apenas, ou adicionar `keyboard,strings`)
- **Subir `INFERENCE_STEPS`** para 50 (melhor qualidade, mais lento)
- **Fixar `SEED`** para reproduzir resultado anterior